# ai_decide reconciliation — blind evaluation (spec 12)

The pipeline asks `ai_decide` to place ONS plants whose CEG doesn't link to SIGA. To measure how far
to trust it, we replay the same question on plants whose answer we **know**: ONS plants that *do* match
SIGA by CEG. The CEG registry number is hidden (so `mesmo_numero_ceg` is unknown); the model sees names, name
similarity, UF, plant type, municipality and capacity,
with the same top-3 candidate builder as the pipeline (`reco_ai.candidates`). The true plant may be
A, B, C, or absent (then the right answer is `nenhum`).

Outputs: accuracy, accuracy and coverage at the accept cutoff (`AI_CUTOFF`), and a calibration table.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../src/transformations"))
import reco_ai  # noqa: E402
import reco_lib as R  # noqa: E402
from pyspark.sql import functions as F  # noqa: E402

dbutils.widgets.text("catalog", "ampera_catalog")  # noqa: F821
dbutils.widgets.text("n", "200")  # noqa: F821
CAT, N = dbutils.widgets.get("catalog"), int(dbutils.widgets.get("n"))  # noqa: F821
print(f"catalog={CAT}  sample={N}  AI_CUTOFF={R.AI_CUTOFF}")

catalog=ampera_catalog  sample=200  AI_CUTOFF=0.9


In [ ]:
ons = spark.table(f"{CAT}.ampera_silver.ons_plant").filter("ons_grain = 'plant'")  # noqa: F821
siga = spark.table(f"{CAT}.ampera_silver.siga_plant")  # noqa: F821
known = (ons.join(siga.select(F.col("plant_id").alias("ceg_norm")), "ceg_norm", "left_semi")
            .withColumn("truth_id", F.col("ceg_norm"))
            .orderBy(F.xxhash64("ons_key")).limit(N)
            # blind: keep only the CEG's type prefix (UTE, EOL...), hide the registry number that gives the answer away
            .withColumn("ceg_raw", F.split("ceg_raw", "[.]")[0]))
c = reco_ai.candidates(known, siga).join(known.select("ons_key", "truth_id"), "ons_key")
truth = (F.when(F.col("a_id") == F.col("truth_id"), "A").when(F.col("b_id") == F.col("truth_id"), "B")
          .when(F.col("c_id") == F.col("truth_id"), "C").otherwise("nenhum"))
# Serverless has no .cache(): persist once so every metric reads the same ai_decide answers.
EVAL = f"{CAT}.ampera_silver.ai_decide_eval"
reco_ai.decide(c.withColumn("truth", truth)).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(EVAL)
spark.sql(f"COMMENT ON TABLE {EVAL} IS 'Silver (GenAI eval): blind ai_decide answers on CEG-matched ONS plants with the CEG hidden. Spec 12.'")  # noqa: F821
res = spark.table(EVAL)  # noqa: F821
print(f"{res.count()} plants scored -> {EVAL}")

200 plants scored -> ampera_catalog.ampera_silver.ai_decide_eval


In [ ]:
def show(df):
    print(df.toPandas().to_string(index=False))


ok = F.col("ai_choice") == F.col("truth")
print("Truth distribution (where the real plant sits among the candidates):")
show(res.groupBy("truth").count().orderBy("truth"))
print("\nOverall:")
show(res.agg(F.count("*").alias("n"), F.round(F.avg(ok.cast("double")), 3).alias("accuracy"),
             F.count_if(F.col("ai_error").isNotNull()).alias("errors")))
print(f"\nAt the accept cutoff (confidence >= {R.AI_CUTOFF}, choice != nenhum) — what the pipeline would auto-apply:")
acc = res.filter("ai_accepted")
show(acc.agg(F.count("*").alias("auto_applied"),
             F.round(F.count("*") / F.lit(res.filter("truth != 'nenhum'").count()), 3).alias("coverage_of_matchable"),
             F.round(F.avg(ok.cast("double")), 3).alias("precision")))

Truth distribution (where the real plant sits among the candidates):
 truth  count
     A    150
     B     10
     C      4
nenhum     36

Overall:
  n  accuracy  errors
200      0.91       0

At the accept cutoff (confidence >= 0.9, choice != nenhum) — what the pipeline would auto-apply:
 auto_applied  coverage_of_matchable  precision
           93                  0.567        1.0


 ## Calibration: does confidence mean what it says?

In [ ]:
b = F.least(F.floor(F.col("ai_confidence") * 10) / 10, F.lit(0.9))
show(res.groupBy(b.alias("confidence_from"))
        .agg(F.count("*").alias("n"), F.round(F.avg("ai_confidence"), 3).alias("mean_confidence"),
             F.round(F.avg(ok.cast("double")), 3).alias("accuracy"))
        .orderBy("confidence_from"))
print("\nConfusion (truth x choice):")
show(res.groupBy("truth").pivot("ai_choice", ["A", "B", "C", "nenhum"]).count().orderBy("truth"))

 confidence_from  n  mean_confidence  accuracy
             0.1  3            0.160     0.000
             0.2  3            0.217     0.667
             0.3 10            0.338     0.400
             0.4 11            0.447     0.727
             0.5 13            0.568     0.846
             0.6 16            0.649     0.875
             0.7 21            0.740     0.952
             0.8 30            0.838     1.000
             0.9 93            0.945     1.000

Confusion (truth x choice):
 truth   A   B   C  nenhum
     A 145 1.0 NaN     4.0
     B   1 9.0 NaN     NaN
     C   1 NaN 1.0     2.0
nenhum   5 3.0 1.0    27.0


 ## Examples: wrong answers (what the model got confused by)

In [ ]:
show(res.filter(~ok).select("nom_usina", "uf", "truth", "ai_choice", F.round("ai_confidence", 2).alias("conf"),
                            "a_nome", "b_nome", "c_nome").limit(15))

     nom_usina uf  truth ai_choice  conf                      a_nome                               b_nome                             c_nome
      Manauara AM nenhum         B  0.14                    Carauari                           MANACAPURU                              Anamã
      Manauara AM nenhum         B  0.18                    Carauari                           MANACAPURU                              Anamã
       Poraque AM nenhum         A  0.42                     Jaraqui                  UGH Parauá Energias                UGH Parauá Energias
 UT.PARNAIBA 4 MA      A         B  0.22                 Parnaíba IV Parnaíba V (Antiga Parnaíba 5A e 5B)                          Delta 4 I
    Goiânia II GO      A    nenhum  0.30                  Goiânia II                          JBS GOIÂNIA                          Goianésia
  UTE CARAMURU GO      C         A  0.16            Caramuru Ipameri                          Cereal Ouro                 Caramuru Itumbiara
 UTE FIBRIA-M

 ## Production run: what the pipeline decided (`ampera_silver.plant_match_ai`)

In [ ]:
show(spark.sql(f"""SELECT ai_choice, ai_accepted, count(*) AS ons_plants, round(avg(ai_confidence), 3) AS mean_conf
                   FROM {CAT}.ampera_silver.plant_match_ai GROUP BY ALL ORDER BY ALL"""))
show(spark.sql(f"""SELECT match_method, count(*) AS ons_series FROM {CAT}.ampera_silver.plant_match
                   GROUP BY ALL ORDER BY ons_series DESC"""))

ai_choice  ai_accepted  ons_plants  mean_conf
        A        False           5      0.318
        A         True           3      0.960
   nenhum        False          25      0.593
 match_method  ons_series
          ceg         408
 ons_conjunto         291
ons_aggregate         234
    unmatched          30
        fuzzy           2
    ai_decide           2
